# Netflix Stock Prediction — End-to-End Analysis

This notebook walks through the full data science pipeline:
1. Data loading & exploration
2. Feature engineering
3. Model training & evaluation
4. SHAP explainability
5. Backtesting

> **Goal:** Predict the next-day return (%) of Netflix stock using technical indicators and an ensemble ML model.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
print('Setup complete')

## 1. Load & Explore Data

In [ ]:
from src.data_loader import load_data
from src.preprocessing import preprocess_data

df_raw = load_data(source='csv')   # change to 'live' for yfinance
df = preprocess_data(df_raw)
print(f'Shape: {df.shape}')
print(f'Date range: {df.index.min().date()} → {df.index.max().date()}')
df.head()

In [ ]:
df.describe().round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(df.index, df['Close'], linewidth=0.8)
axes[0].set_title('Netflix Close Price — Full History')

ret = df['Close'].pct_change().dropna() * 100
axes[1].hist(ret, bins=100, color='coral', edgecolor='none')
axes[1].set_title('Daily Returns Distribution (%)')
plt.tight_layout()

## 2. Feature Engineering

In [ ]:
from src.feature_engineering import create_features
from src.modeling import FEATURES

df = create_features(df)
print(f'Features: {len(FEATURES)}')
print(f'Rows after dropna: {len(df):,}')
df[FEATURES].describe().round(3)

In [ ]:
# Correlation of features with next-day return
df['NextReturn'] = df['Close'].pct_change(-1) * 100
corr = df[FEATURES + ['NextReturn']].corr()['NextReturn'].drop('NextReturn').sort_values()

fig, ax = plt.subplots(figsize=(8, 12))
corr.plot(kind='barh', ax=ax, color=['red' if v < 0 else 'green' for v in corr])
ax.set_title('Feature Correlation with Next-Day Return')
ax.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()

## 3. Model Training & Evaluation

In [ ]:
from src.modeling import train_model, save_model

model, results, X_test, y_test, preds = train_model(df)

print('\n📊 Test Set Results:')
for k, v in results.items():
    print(f'  {k}: {v:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(np.array(y_test), label='Actual', linewidth=1)
axes[0].plot(preds, label='Predicted', linewidth=1, linestyle='--', alpha=0.8)
axes[0].set_title(f"Actual vs Predicted  |  R²={results['R2']:.4f}")
axes[0].legend()

residuals = np.array(y_test) - preds
axes[1].hist(residuals, bins=60, color='steelblue', edgecolor='none')
axes[1].set_title('Residual Distribution')
plt.tight_layout()

## 4. SHAP Explainability

In [ ]:
import shap

xgb_est = None
for name, est in model.fitted_learners_:
    if 'xgb' in name:
        xgb_est = est
        break

X_scaled = model.scaler.transform(X_test)
explainer = shap.TreeExplainer(xgb_est)
shap_values = explainer.shap_values(X_scaled[:200])

shap.summary_plot(shap_values, X_scaled[:200], feature_names=FEATURES)

## 5. Backtesting

In [ ]:
from src.backtest import run_backtest

y_test_ret = (y_test.values - X_test['Lag1'].values) / X_test['Lag1'].values * 100
pred_ret   = preds - X_test['Lag1'].values

bt = run_backtest(y_test_ret, pred_ret)

print('\n📊 Backtest Metrics:')
for k, v in bt['metrics'].items():
    print(f'  {k}: {v}')

In [ ]:
curves = bt['curves']
plt.figure(figsize=(12, 5))
plt.plot(curves.index, curves['Strategy'],   label='Model Strategy', linewidth=1.5)
plt.plot(curves.index, curves['BuyAndHold'], label='Buy & Hold',     linewidth=1.5, linestyle='--')
plt.axhline(1.0, color='gray', linestyle=':', alpha=0.5)
plt.title('Strategy vs Buy & Hold — Equity Curve')
plt.ylabel('Portfolio Value (start = 1.0)')
plt.legend()
plt.tight_layout()

## Summary

| Component | Detail |
|---|---|
| Model | XGB + LGBM + RF + ET → Ridge (manual stacking) |
| Features | 49 technical indicators |
| Target | Next-day return (%) |
| Validation | 5-fold walk-forward CV |
| Key metric | Directional Accuracy (>52% = meaningful signal) |

> Stock prediction is hard. This project demonstrates a rigorous ML pipeline — the real value is in the methodology, not the absolute prediction accuracy.